# AHP Criteria Weight Calculator / حاسبة أوزان المعايير بطريقة التحليل الهرمي

This notebook computes the weights of the seven decision criteria from an AHP pairwise-comparison survey exported from Google Forms.
تحسب هذه الأداة أوزان معايير القرار السبعة من استبيان المقارنات الزوجية (AHP) المُصدَّر من Google Forms.

**Method / المنهجية:**
- Each response is converted from the 1–9 Google Forms scale to Saaty intensities. / تحويل مقياس 1–9 إلى قيم Saaty.
- A reciprocal comparison matrix is built per respondent; weights are derived by the eigenvector method. / بناء مصفوفة لكل مستجيب واشتقاق الأوزان بالمتّجه الذاتي.
- Responses with high inconsistency (CR ≥ 0.20) are excluded. / استبعاد الاستجابات ذات التناقض المرتفع.
- Remaining judgments are aggregated by the geometric mean (Aczél & Saaty, 1983). / دمج الأحكام بالمتوسط الهندسي.
- The consistency of the aggregated matrix is verified (CR < 0.10). / التحقّق من اتساق المصفوفة المجمّعة.

## Step 1 — Upload the responses CSV / رفع ملف الإجابات

In [ ]:
from google.colab import files
import pandas as pd, io
uploaded = files.upload()
fname = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[fname]))
print(f"Responses loaded / عدد الاستجابات: {len(df)}")
df.head()

## Step 2 — Identify the 21 comparison columns / تحديد أعمدة المقارنات
Google Forms adds leading columns (timestamp, participant info); the 21 comparison columns follow them.
يضيف Google Forms أعمدةً في البداية (الطابع الزمني ومعلومات المشارك)، وتليها أعمدة المقارنات الـ21.

In [ ]:
for i, col in enumerate(df.columns):
    print(f"{i}: {col[:70]}")

In [ ]:
# Index of the first comparison column / رقم أول عمود مقارنة
FIRST_Q_COL = 4
comparison_cols = df.columns[FIRST_Q_COL : FIRST_Q_COL + 21]
assert len(comparison_cols) == 21, f"Expected 21 columns, found {len(comparison_cols)}"
print("Comparison columns identified / تمّ تحديد أعمدة المقارنات:", len(comparison_cols))

## Step 3 — AHP logic / منطق التحليل الهرمي

In [ ]:
import numpy as np

CRIT    = ["التوافرية","زمن الاسترجاع","التكلفة","الامتثال","قابلية النقل","الأمن والحوكمة","التعقيد"]
CRIT_EN = ["Availability","RTO","Cost","Compliance","Portability","Security","Complexity"]
KEY     = ["avail","rto","cost","compliance","portability","security","complexity"]
PAIRS   = [(0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(1,2),(1,3),(1,4),(1,5),(1,6),
           (2,3),(2,4),(2,5),(2,6),(3,4),(3,5),(3,6),(4,5),(4,6),(5,6)]
RI = {3:0.58, 4:0.90, 5:1.12, 6:1.24, 7:1.32}   # Saaty random index
CR_THRESHOLD = 0.20   # responses with CR at or above this value are excluded

def gforms_to_saaty(v):
    """Convert a 1-9 Google Forms value to a signed Saaty intensity.
    Midpoint 5 = equal; toward 1 the first criterion dominates; toward 9 the second."""
    m = {5:1, 4:3, 3:5, 2:7, 1:9, 6:3, 7:5, 8:7, 9:9}
    intensity = m[int(round(v))]
    return intensity if v <= 5 else -intensity

def build_matrix(answers):
    """Build a 7x7 reciprocal comparison matrix from 21 answers."""
    n = len(CRIT); M = np.ones((n, n))
    for k, (i, j) in enumerate(PAIRS):
        s = gforms_to_saaty(answers[k]); val = abs(s)
        if s >= 0: M[i, j] = val; M[j, i] = 1/val
        else:      M[j, i] = val; M[i, j] = 1/val
    return M

def analyze(M):
    """Return the weight vector (eigenvector method) and the consistency ratio CR."""
    n = M.shape[0]
    eigvals, eigvecs = np.linalg.eig(M)
    k = np.argmax(eigvals.real)
    w = np.abs(eigvecs[:, k].real); w = w / w.sum()
    lam_max = eigvals[k].real
    CI = (lam_max - n) / (n - 1)
    return w, CI / RI[n]

def geometric_mean(matrices):
    """Aggregate several comparison matrices element-wise by the geometric mean."""
    prod = np.ones_like(matrices[0])
    for M in matrices: prod = prod * M
    return prod ** (1 / len(matrices))

print("AHP logic ready / المنطق جاهز")

## Step 4 — Per-response consistency / فحص اتساق كل استجابة

In [ ]:
matrices = []; ratios = []
for idx, row in df.iterrows():
    answers = [row[c] for c in comparison_cols]
    if any(pd.isna(a) for a in answers):
        print(f"Response {idx+1}: incomplete, skipped / ناقصة، تُتجاوز"); continue
    M = build_matrix(answers); _, CR = analyze(M)
    matrices.append(M); ratios.append(CR)
    print(f"Response {idx+1}: CR = {CR:.3f}")

ratios = np.array(ratios)
print(f"\nTotal / الإجمالي: {len(ratios)}  |  CR < 0.20: {(ratios < CR_THRESHOLD).sum()}")

## Step 5 — Aggregate consistent responses / دمج الاستجابات المتّسقة
Responses with CR below the threshold are aggregated by the geometric mean, and the consistency of the group matrix is verified.
تُدمَج الاستجابات ذات التناقض المنخفض بالمتوسط الهندسي، ويُتحقَّق من اتساق مصفوفة المجموعة.

In [ ]:
kept = [M for M, cr in zip(matrices, ratios) if cr < CR_THRESHOLD]
print(f"Responses aggregated / عدد الاستجابات المدموجة: {len(kept)}")

M_group = geometric_mean(kept)
weights, CR_group = analyze(M_group)

print(f"Aggregated matrix CR / اتساق المصفوفة المجمّعة: {CR_group:.4f}", 
      "(consistent / متّسقة)" if CR_group < 0.10 else "")
print("="*52)
print(f"{'Criterion / المعيار':<30}{'Weight / الوزن':>12}")
print("-"*52)
order = np.argsort(-weights)
for i in order:
    print(f"{CRIT_EN[i]+' / '+CRIT[i]:<30}{weights[i]*100:>10.1f}%")
print("="*52)

WEIGHTS = {KEY[i]: round(float(weights[i]), 3) for i in range(len(KEY))}
# adjust rounding so the weights sum to exactly 1.0
drift = round(1.0 - sum(WEIGHTS.values()), 3)
if drift:
    kmax = max(WEIGHTS, key=WEIGHTS.get); WEIGHTS[kmax] = round(WEIGHTS[kmax] + drift, 3)
print("WEIGHTS =", WEIGHTS)

## References / المراجع
- Saaty, T. L. (1980). *The Analytic Hierarchy Process.* McGraw-Hill.
- Aczél, J., & Saaty, T. L. (1983). Procedures for synthesizing ratio judgements. *Journal of Mathematical Psychology, 27*(1), 93–102.